## Evaluate Presidio Analyzer using the Presidio Evaluator framework

This notebook demonstrates how to evaluate a Presidio instance using the presidio-evaluator framework.

Steps:
1. Load dataset
2. Dataset statistics
3. Define the Presidio Analyzer
4. Run predictions
5. Review and adjust entity mapping
6. Evaluate
7. Results and error analysis

For an example with a custom Presidio instance, see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

In [1]:
# install presidio evaluator via pip if not yet installed

#!pip install presidio-evaluator

In [2]:
import json
from collections import Counter
from pathlib import Path
from pprint import pprint
import pandas as pd
from presidio_analyzer import AnalyzerEngine

from presidio_evaluator import InputSample
from presidio_evaluator.entity_mapping import CanonicalMapper, IncompleteMapping
from presidio_evaluator.evaluation import ModelError, Plotter, SpanEvaluator
from presidio_evaluator.experiment_tracking import get_experiment_tracker
from presidio_evaluator.models import PresidioAnalyzerWrapper

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


## 1. Load dataset from file

In [3]:
dataset_name = "generated_size_1500_date_October_21_2026.json"
dataset = InputSample.read_dataset_json(Path(Path.cwd().parent, "data", dataset_name))
print(len(dataset))

tokenizing input:   0%|          | 1/1500 [00:00<04:32,  5.51it/s]

loading model en_core_web_sm


tokenizing input: 100%|██████████| 1500/1500 [00:04<00:00, 339.42it/s]

1500


This dataset was auto generated. See more info here [Synthetic data generation](1_Generate_data.ipynb).

In [4]:
def get_entity_counts(dataset: list[InputSample]) -> Counter:
    """Return a dictionary with counter per entity type."""
    entity_counter = Counter()
    for sample in dataset:
        for tag in sample.tags:
            entity_counter[tag] += 1
    return entity_counter

## 2. Simple dataset statistics

In [5]:
entity_counts = get_entity_counts(dataset)
print("Count per entity:")
pprint(entity_counts.most_common(), compact=True)

print(
    "\nMin and max number of tokens in dataset: "
    f"Min: {min([len(sample.tokens) for sample in dataset])}, "
    f"Max: {max([len(sample.tokens) for sample in dataset])}"
)

print(
    f"Min and max sentence length in dataset: "
    f"Min: {min([len(sample.full_text) for sample in dataset])}, "
    f"Max: {max([len(sample.full_text) for sample in dataset])}"
)

print("\nExample InputSample:")
print(dataset[0])

Count per entity:
[('O', 18112), ('STREET_ADDRESS', 2978), ('PERSON', 1425),
 ('ORGANIZATION', 818), ('GPE', 439), ('PHONE_NUMBER', 424), ('DATE_TIME', 194),
 ('CREDIT_CARD', 110), ('UK_NINO', 96), ('TITLE', 88), ('AGE', 80), ('NRP', 48),
 ('US_SSN', 48), ('EMAIL_ADDRESS', 42), ('UK_DRIVING_LICENCE', 35),
 ('ZIP_CODE', 35), ('UK_PASSPORT', 33), ('DOMAIN_NAME', 31), ('IBAN_CODE', 27),
 ('IP_ADDRESS', 15), ('US_DRIVER_LICENSE', 6)]

Min and max number of tokens in dataset: Min: 2, Max: 80
Min and max sentence length in dataset: Min: 9, Max: 409

Example InputSample:
Full text: You can reach me on 01287 982941
Spans: [Span(type: PHONE_NUMBER, value: 01287 982941, char_span: [20: 32])]



## 3. Define the Presidio Analyzer

Using Presidio with default parameters (not recommended for production). For a customised example see [notebook 5](5_Evaluate_Custom_Presidio_Analyzer.ipynb).

**Note:** The dataset may use different entity labels than Presidio (e.g. `STREET_ADDRESS` vs `LOCATION`). We'll align them in section 5 after running predictions.

In [6]:
# Loading the vanilla Analyzer Engine, with the default NER model.
analyzer_engine = AnalyzerEngine(default_score_threshold=0.4)

pprint("Supported entities for English:")
pprint(analyzer_engine.get_supported_entities("en"), compact=True)

print("\nLoaded recognizers for English:")
pprint(
    [rec.name for rec in analyzer_engine.registry.get_recognizers("en", all_fields=True)],
    compact=True,
)

print("\nLoaded NER models:")
pprint(analyzer_engine.nlp_engine.models)

'Supported entities for English:'
['CREDIT_CARD', 'US_DRIVER_LICENSE', 'MAC_ADDRESS', 'URL', 'UK_NHS', 'US_SSN',
 'NRP', 'LOCATION', 'CRYPTO', 'PHONE_NUMBER', 'US_BANK_NUMBER', 'DATE_TIME',
 'US_ITIN', 'MEDICAL_LICENSE', 'US_PASSPORT', 'IBAN_CODE', 'PERSON',
 'EMAIL_ADDRESS', 'IP_ADDRESS']

Loaded recognizers for English:
['CreditCardRecognizer', 'UsBankRecognizer', 'UsLicenseRecognizer',
 'UsItinRecognizer', 'UsPassportRecognizer', 'UsSsnRecognizer', 'NhsRecognizer',
 'CryptoRecognizer', 'DateRecognizer', 'EmailRecognizer', 'IbanRecognizer',
 'IpRecognizer', 'MedicalLicenseRecognizer', 'MacAddressRecognizer',
 'PhoneRecognizer', 'UrlRecognizer', 'SpacyRecognizer']

Loaded NER models:
[{'lang_code': 'en', 'model_name': 'en_core_web_lg'}]


In [7]:
# Wrap the analyzer for evaluation and analysis
wrapped_analyzer = PresidioAnalyzerWrapper(analyzer_engine=analyzer_engine)

# Set up the experiment tracker and log model + dataset params
experiment = get_experiment_tracker()
params = {"dataset_name": dataset_name, "model_name": wrapped_analyzer.name}
params.update(wrapped_analyzer.to_log())
experiment.log_parameters(params)
experiment.log_dataset_hash(dataset)

## 4. Run predictions

Run the model on the full dataset. Returns a 5-column DataFrame (`sentence_id`, `token`, `annotation`, `prediction`, `start_indices`).

In [8]:
%%time
results_df = wrapped_analyzer.predict_dataset(dataset)
results_df.head()

CPU times: user 5.24 s, sys: 81 ms, total: 5.32 s
Wall time: 5.36 s


,sentence_id,token,annotation,prediction,start_indices,annotation_span_id,prediction_span_id
0,0,You,O,O,0,None,None
1,0,can,O,O,4,None,None
2,0,reach,O,O,8,None,None
3,0,me,O,O,14,None,None
4,0,on,O,O,17,None,None


## 5. Review entity mapping

- The entity mapping process auto-resolves labels using a hierarchical entity tree.
The `CanonicalMapper` class analyzes the results (predictions and annotations), and checks if there are any gaps towards the evaluation process.

- If you call `render_html()` to inspect issues, you'd see if there are any gaps that prevent you from completing the evaluation successfully. Make sure you address all the issues by using:
  -  `mapper.map({"LABEL": "CANONICAL"})` to remap or 
  -  `mapper.map({"LABEL": None})` to suppress.

- For more information about entity mapping, see [Notebook 6 - Entity Mapping](6_Interactive_Entity_Mapping.ipynb).

In [9]:
mapper = CanonicalMapper()
mapper.analyze(results_df)
mapper.render_html()

Label,Resolved as,Tokens
STREET_ADDRESS,ADDRESS,2978
PERSON,PERSON,1425
ORGANIZATION,ORGANIZATION,818
GPE,GPE,439
PHONE_NUMBER,PHONE_NUMBER,424
DATE_TIME,DATE_TIME,194
CREDIT_CARD,FINANCIAL,110
UK_NINO,SSN,96
TITLE,TITLE,88
AGE,AGE,80


In [10]:
mapped_results = mapper.get_mapped_results_dataframe()

# Log entity mappings for experiment tracking
experiment.log_parameter('entity_mappings', json.dumps(mapper.get_mapping()))

## 6. Evaluate

In [11]:
evaluator = SpanEvaluator(iou_threshold=0.75)
f_beta = 2  # F-beta parameter used throughout evaluation


skip words not provided, using default skip words. If you want the evaluation to not use skip words, pass skip_words=[]


In [12]:
hierarchical_results = evaluator.calculate_hierarchical_scores(
    mapped_results, 
    beta=f_beta
    )
binary_results = hierarchical_results["binary"]
branch_results = hierarchical_results["branch"]
detailed_results = hierarchical_results["detailed"]


# A. Binary PII/O metrics
print("Level: binary (PII vs O):")
pprint({
    "Precision": round(binary_results.pii_precision, 3),
    "Recall":    round(binary_results.pii_recall, 3),
    f"F{f_beta}":  round(binary_results.pii_f, 3),
})

Level: binary (PII vs O):
{'F2': 0.565, 'Precision': 0.783, 'Recall': 0.528}


In [13]:
# B. Branch-level per-entity scores (PERSON, LOCATION, DATE_TIME, …)
branch_plotter = Plotter(
    results=branch_results,
    model_name=wrapped_analyzer.name,
    display_mode="interactive",
    beta=f_beta,
)
branch_plotter.plot_scores(include_pii_aggregate=False, annotation_entities_only=True)

In [14]:
# C. Detailed-level per-entity scores (NAME, GPE, PHONE_NUMBER, …)
detailed_plotter = Plotter(
    results=detailed_results,
    model_name=wrapped_analyzer.name,
    display_mode="interactive",
    beta=f_beta,
)
detailed_plotter.plot_scores(include_pii_aggregate=False, annotation_entities_only=True)

## 7. Confusion matrix

In [15]:
# D. Branch-level confusion matrix (only branch entities that appear in the data)
entities, confmatrix = branch_results.to_confusion_matrix()
branch_plotter.plot_confusion_matrix(entities=entities, confmatrix=confmatrix)

## 8. Error analysis

Now let's look into results to understand what's behind the metrics we're getting.
Note that evaluation is never perfect. Some things to consider:
1. There's often a mismatch between the annotated span and the predicted span, which isn't necessarily a mistake. For example: `<Southern France>` compared with `Southern <France>`. In the second text, the word `Southern` was not annotated/predicted as part of the entity, but that's not necessarily an error.
1. The synthetic dataset used here isn't representative of a real dataset. Consider using more realistic datasets for evaluation

In [16]:
branch_plotter.plot_most_common_tokens()

### 8a. False positives
#### Most common false positive tokens:

In [17]:
ModelError.most_common_fp_tokens(branch_results.model_errors)

Most common false positive tokens:
[('couple', 26),
 ('8', 22),
 ('british', 18),
 ('10th', 18),
 ('white stripes', 16),
 ('60s', 16),
 ('sunday labor', 14),
 ('answer:"tube snake boogie', 12),
 ('uk', 10),
 ('5 pm', 10)]
---------------
Example sentence with each FP token:
	- a couple of months (`couple` pred as DATE_TIME)
	- 8 + years (`8` pred as DATE_TIME)
	- British (`british` pred as DEMOGRAPHIC)
	- 10th year (`10th` pred as DATE_TIME)
	- the White Stripes ' (`white stripes` pred as LOCATION)
	- the 60s (`60s` pred as DATE_TIME)
	- the last Sunday before Labor Day (`sunday labor` pred as DATE_TIME)
	- Answer:"Tube Snake Boogie (`answer:"tube snake boogie` pred as PERSON)
	- UK (`uk` pred as LOCATION)
	- 5 pm (`5 pm` pred as DATE_TIME)


[('couple', 26),
 ('8', 22),
 ('british', 18),
 ('10th', 18),
 ('white stripes', 16),
 ('60s', 16),
 ('sunday labor', 14),
 ('answer:"tube snake boogie', 12),
 ('uk', 10),
 ('5 pm', 10)]

#### More FP analysis

In [18]:
# Get false positives for PERSON entity
fps_df = ModelError.get_fps_dataframe(branch_results.model_errors, entity="PERSON")
fps_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,Greenlander,greenlander,O,PERSON
1,Nemetschek,nemetschek,O,PERSON
2,Rue Galboun Ibn Al Hassen Apt,galboun ibn al hassen,O,PERSON
3,Horjul,horjul,O,PERSON
4,Grosse Praesidenten Str,grosse praesidenten str,O,PERSON
5,Miia Uotila,miia uotila,O,PERSON
6,Manlio,manlio,O,PERSON
7,Son,son,O,PERSON
8,Moura,moura,O,PERSON
9,Abdulton,abdulton,O,PERSON


### 8b. False negatives (FN)

#### Most common false negative examples (should be predicted as entity but wasn't):

In [19]:
ModelError.most_common_fn_tokens(branch_results.model_errors, n=15)

Most common false negative tokens:
[('mr.', 17),
 ('greenlander', 9),
 ('63', 9),
 ('35', 8),
 ('mr', 7),
 ('50', 7),
 ('61', 6),
 ('77', 6),
 ('48', 6),
 ('33', 5),
 ('82', 5),
 ('34', 4),
 ('78', 4),
 ('46', 4),
 ('24', 4)]
---------------
Example sentence with each FN token:
	- Mr. (`mr.` annotated as PERSON)
	- Greenlander (`greenlander` annotated as LOCATION)
	- 63 (`63` annotated as DEMOGRAPHIC)
	- 35 (`35` annotated as DEMOGRAPHIC)
	- Mr (`mr` annotated as PERSON)
	- 50 (`50` annotated as DEMOGRAPHIC)
	- 61 (`61` annotated as DEMOGRAPHIC)
	- 77 (`77` annotated as DEMOGRAPHIC)
	- 48 (`48` annotated as DEMOGRAPHIC)
	- 33 (`33` annotated as DEMOGRAPHIC)
	- 82 (`82` annotated as DEMOGRAPHIC)
	- 34 (`34` annotated as DEMOGRAPHIC)
	- 78 (`78` annotated as DEMOGRAPHIC)
	- 46 (`46` annotated as DEMOGRAPHIC)
	- 24 (`24` annotated as DEMOGRAPHIC)


[('mr.', 17),
 ('greenlander', 9),
 ('63', 9),
 ('35', 8),
 ('mr', 7),
 ('50', 7),
 ('61', 6),
 ('77', 6),
 ('48', 6),
 ('33', 5),
 ('82', 5),
 ('34', 4),
 ('78', 4),
 ('46', 4),
 ('24', 4)]

#### More FN analysis

In [20]:
# Get false negatives for PHONE_NUMBER entity
fns_df = ModelError.get_fns_dataframe(branch_results.model_errors, entity="CONTACT")

In [21]:
fns_df[["full_text", "token", "annotation", "prediction"]].head(20)

,full_text,token,annotation,prediction
0,01287 982941,01287 982941,CONTACT,O
1,01175 740579,01175 740579,CONTACT,O
2,062 571 54 15,062 571 54 15,CONTACT,O
3,25 833307,25 833307,CONTACT,O
4,041 034 3579\nDesk,041 034 3579\ndesk,CONTACT,O
5,070 8391 6971,070 8391 6971,CONTACT,O
6,07353 024399,07353 024399,CONTACT,O
7,01860 319908,01860 319908,CONTACT,O
8,9845 8544,9845 8544,CONTACT,O
9,0680 931 34 27,0680 931 34 27,CONTACT,O
